# 03. Feature Engineering & Data Leakage Prevention
## Water Quality Prediction via Gradient Boosting & Deep Neural Networks (Phase 1)

### Objectives:
- Engineer physically grounded chemical interaction features (DO saturation ratio, BOD/DO organic stress, nitrogen speciation, eutrophication index).
- Execute an automated data leakage audit ($|r| \ge 0.98$ checks) against both continuous and discrete targets.
- Ensure absolute mathematical isolation using `SplitAwareImputer` (fit-on-train-only).
- Formulate site-separated (GroupKFold by `Area`) and stratified cross-validation strategies.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.features.engineering import (
    compute_do_saturation,
    audit_data_leakage,
    ChemicalDomainFeatureExtractor,
    SplitAwareImputer
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print("Feature engineering environment initialized!")

### 1. Ingest Data

In [ ]:
dataset_path = ROOT_DIR / "data" / "raw" / "Combined_dataset.csv"
df = pd.read_csv(dataset_path, nrows=75000)
print(f"Loaded {len(df):,} samples for feature engineering verification.")

### 2. Physics-Informed Feature Engineering: Chemical Domain Transformer
Water quality indices represent non-linear interactions between multiple chemical species. We construct features representing:
1. **DO Saturation Ratio & Deficit**: Temperature-dependent theoretical oxygen solubility.
2. **BOD-to-DO Ratio**: Organic matter microbial consumption vs available dissolved oxygen.
3. **Nitrogen Speciation**: Fractions of toxic un-ionized ammonia vs oxidized nitrate.
4. **Nutrient Enrichment Product**: Combined nitrate-phosphate loading causing algal blooms.

In [ ]:
fe = ChemicalDomainFeatureExtractor()
df_engineered = fe.transform(df)

new_features = [c for c in df_engineered.columns if c.startswith('feat_')]
print(f"Successfully engineered {len(new_features)} domain features:")
display(df_engineered[new_features].describe().T[['mean', 'std', 'min', 'max']])

### 3. Visualizing Key Chemical Interactions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature vs DO Saturation Ratio
sns.scatterplot(
    data=df_engineered.sample(5000, random_state=42),
    x='Temperature (cel)', y='feat_do_saturation_ratio',
    hue='CCME_WQI', alpha=0.5, palette='viridis', ax=axes[0]
)
axes[0].set_title('Temperature vs DO Saturation Ratio by Quality Tier', fontweight='bold')
axes[0].set_ylim(0, 2.5)

# BOD to DO Ratio vs CCME Values
sns.scatterplot(
    data=df_engineered.sample(5000, random_state=42),
    x='feat_bod_to_do_ratio', y='CCME_Values',
    alpha=0.5, color='#d62728', ax=axes[1]
)
axes[1].set_title('Organic Stress (BOD/DO Ratio) vs CCME Score', fontweight='bold')
axes[1].set_xlim(0, 10)

plt.tight_layout()
plt.show()

### 4. Rigorous Data Leakage Audit
A frequent error in environmental ML is target leakage: when the target WQI is computed via an index formula from features present in the dataset, the model can trivially memorize the formula ($R^2 \approx 1.0$) rather than learning generalizable environmental patterns.

In [ ]:
# Audit correlation against continuous target CCME_Values
audit_res = audit_data_leakage(df_engineered.drop(columns=['CCME_WQI']), target_col='CCME_Values', threshold=0.98)
print("Top 5 Feature Correlations with CCME_Values:")
for feat, corr in audit_res['top_correlated_features']:
    print(f"  {feat:35s}: Pearson r = {corr:+.4f}")

if audit_res['flagged_leakages']:
    print("\nWARNING: Potential leakage flagged:", audit_res['flagged_leakages'])
else:
    print("\nSUCCESS: Data leakage audit PASSED. No single feature has |r| >= 0.98.")

### 5. Split-Aware Imputation (Zero Cross-Split Lookahead Bias)
Imputing missing values across the entire dataset before splitting leaks distribution parameters of the holdout set into training. We verify that `SplitAwareImputer` fits exclusively on `X_train`.

In [ ]:
from sklearn.model_selection import train_test_split

features_only = [c for c in df_engineered.columns if c not in ['Country', 'Area', 'Waterbody Type', 'Date', 'CCME_Values', 'CCME_WQI']]
X = df_engineered[features_only]
y = df_engineered['CCME_Values']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

imputer = SplitAwareImputer(strategy='median', add_indicator=True)
imputer.fit(X_train)
X_train_imp = imputer.transform(X_train)
X_test_imp = imputer.transform(X_test)

print(f"Train imputed shape: {X_train_imp.shape} | Nulls: {X_train_imp.isna().sum().sum()}")
print(f"Test imputed shape:  {X_test_imp.shape} | Nulls: {X_test_imp.isna().sum().sum()}")
print("Split-aware imputation verified successfully!")